# 点机制放置：连续位置、CV 归属与运行时实例身份

本 notebook 说明 BrainCell 将连续形态位置映射到离散电模型时，如何保留每一次点机制放置为独立实例。需要区分以下身份：

- `branch_id + branch_x`：原始连续位置声明及形态学归属。
- `cv_id + cv_position`：唯一的控制体积（CV）归属。
- `point_id`：读取膜电位、汇入点电流的电学节点。
- `placement_id`：独立点机制实例的身份。
- `synapse_index`：projection 在某个突触 layout 的 placement 轴上的局部索引。

多个 placement 可以共享同一 CV、同一电学 point 和同一个向量化运行时 layout，但它们不会因此合并。

In [1]:
import brainunit as u
import numpy as np
import pandas as pd

import braincell
from braincell.filter import at
from braincell.network import EdgeSet, Projection, by_post

## 辅助函数

每个例子都创建一个全新的 cell。检查函数既输出普通的 `list[dict]`，也将其显示为表格。

In [2]:
def build_morphology(with_dend=False):
    soma = braincell.Branch.from_lengths(
        lengths=[20.0] * u.um,
        radii=[10.0, 10.0] * u.um,
        type="soma",
    )
    morpho = braincell.Morphology.from_root(soma, name="soma")
    if with_dend:
        dend = braincell.Branch.from_lengths(
            lengths=[100.0] * u.um,
            radii=[2.0, 1.0] * u.um,
            type="basal_dendrite",
        )
        morpho.soma.attach(dend, name="dend", parent_x=1.0, child_x=0.0)
    return morpho


def build_cell(with_dend=False, *, size=None):
    pop_size = () if size is None else (size,)
    return braincell.Cell(
        build_morphology(with_dend=with_dend),
        cv_policy=braincell.CVPerBranch(),
        pop_size=pop_size,
        V_init=-65.0 * u.mV,
    )


def exp_synapse(name="exp"):
    return braincell.mech.Synapse(
        "ExpSyn",
        name=name,
        tau=2.0 * u.ms,
        e=0.0 * u.mV,
        weight=0.1 * u.uS,
    )


def placement_rows(cell):
    rows = []
    for item in cell.point_placements:
        rows.append(
            {
                "placement_id": item.id,
                "mechanism": getattr(item.mechanism, "instance_name", type(item.mechanism).__name__),
                "branch_id": item.branch_id,
                "branch_name": item.branch_name,
                "branch_type": item.branch_type,
                "branch_x": item.branch_x,
                "cv_id": item.cv_id,
                "cv_position": item.cv_position,
                "point_id": item.point_id,
            }
        )
    return rows


def layout_rows(cell):
    rows = []
    for layout in cell.layouts:
        placement_ids = (
            [None] * layout.n_active
            if layout.placement_index is None
            else layout.placement_index.tolist()
        )
        point_ids = [None] * layout.n_active if layout.point_index is None else layout.point_index.tolist()
        for local_index, (placement_id, point_id) in enumerate(zip(placement_ids, point_ids)):
            rows.append(
                {
                    "layout_id": layout.id,
                    "kind": layout.kind,
                    "local_index": local_index,
                    "placement_id": placement_id,
                    "point_id": point_id,
                    "n_active": layout.n_active,
                }
            )
    return rows

## 1. 同一 CV 内的不同连续位置不会合并

当 soma 只有一个 CV 时，`soma(0.31)` 和 `soma(0.39)` 都映射到该 CV 的中点电学节点，但仍创建两个 placement。

In [3]:
cell = build_cell()
syn = exp_synapse()
cell.place(at("soma", 0.31), syn)
cell.place(at("soma", 0.39), syn)

rows = placement_rows(cell)
print(rows)
display(pd.DataFrame(rows))

assert [row["branch_x"] for row in rows] == [0.31, 0.39]
assert [row["cv_id"] for row in rows] == [0, 0]
assert len({row["point_id"] for row in rows}) == 1
assert [row["placement_id"] for row in rows] == [0, 1]

[{'placement_id': 0, 'mechanism': 'exp', 'branch_id': 0, 'branch_name': 'soma', 'branch_type': 'soma', 'branch_x': 0.31, 'cv_id': 0, 'cv_position': 'mid', 'point_id': 1}, {'placement_id': 1, 'mechanism': 'exp', 'branch_id': 0, 'branch_name': 'soma', 'branch_type': 'soma', 'branch_x': 0.39, 'cv_id': 0, 'cv_position': 'mid', 'point_id': 1}]


,placement_id,mechanism,branch_id,branch_name,branch_type,branch_x,cv_id,cv_position,point_id
0,0,exp,0,soma,soma,0.31,0,mid,1
1,1,exp,0,soma,soma,0.39,0,mid,1


In [4]:
cell.init_state()
layout = next(layout for layout, _ in cell.runtime.iter_synapse_layouts())
runtime_rows = layout_rows(cell)
print(runtime_rows)
display(pd.DataFrame(runtime_rows))

assert layout.point_index.tolist() == [1, 1]
assert layout.placement_index.tolist() == [0, 1]
assert layout.n_active == 2

[{'layout_id': 0, 'kind': 'synapse:ExpSyn', 'local_index': 0, 'placement_id': 0, 'point_id': 1, 'n_active': 2}, {'layout_id': 0, 'kind': 'synapse:ExpSyn', 'local_index': 1, 'placement_id': 1, 'point_id': 1, 'n_active': 2}]


,layout_id,kind,local_index,placement_id,point_id,n_active
0,0,synapse:ExpSyn,0,0,1,2
1,0,synapse:ExpSyn,1,1,1,2


## 2. 重复完全相同的 `place()` 调用也不会合并

点机制对象是可复用的规格描述；它每次出现在 `place()` 中，都会创建新的 placement 实例。

In [5]:
cell = build_cell()
syn = exp_synapse()
loc = at("soma", 0.5)
cell.place(loc, syn)
cell.place(loc, syn)

rows = placement_rows(cell)
print(rows)
display(pd.DataFrame(rows))
assert len(rows) == 2
assert [row["placement_id"] for row in rows] == [0, 1]
assert len({(row["branch_x"], row["cv_id"], row["point_id"]) for row in rows}) == 1

[{'placement_id': 0, 'mechanism': 'exp', 'branch_id': 0, 'branch_name': 'soma', 'branch_type': 'soma', 'branch_x': 0.5, 'cv_id': 0, 'cv_position': 'mid', 'point_id': 1}, {'placement_id': 1, 'mechanism': 'exp', 'branch_id': 0, 'branch_name': 'soma', 'branch_type': 'soma', 'branch_x': 0.5, 'cv_id': 0, 'cv_position': 'mid', 'point_id': 1}]


,placement_id,mechanism,branch_id,branch_name,branch_type,branch_x,cv_id,cv_position,point_id
0,0,exp,0,soma,soma,0.5,0,mid,1
1,1,exp,0,soma,soma,0.5,0,mid,1


## 3. 在一次 `place()` 中重复机制也不会合并

这是在同一连续位置显式请求多个独立实例的另一种写法。

In [6]:
cell = build_cell()
syn = exp_synapse()
cell.place(at("soma", 0.5), syn, syn)

rows = placement_rows(cell)
print(rows)
display(pd.DataFrame(rows))
assert len(rows) == 2
assert [row["placement_id"] for row in rows] == [0, 1]

[{'placement_id': 0, 'mechanism': 'exp', 'branch_id': 0, 'branch_name': 'soma', 'branch_type': 'soma', 'branch_x': 0.5, 'cv_id': 0, 'cv_position': 'mid', 'point_id': 1}, {'placement_id': 1, 'mechanism': 'exp', 'branch_id': 0, 'branch_name': 'soma', 'branch_type': 'soma', 'branch_x': 0.5, 'cv_id': 0, 'cv_position': 'mid', 'point_id': 1}]


,placement_id,mechanism,branch_id,branch_name,branch_type,branch_x,cv_id,cv_position,point_id
0,0,exp,0,soma,soma,0.5,0,mid,1
1,1,exp,0,soma,soma,0.5,0,mid,1


## 4. 共享 layout 不等于实例合并

相同规格可共享一个向量化 layout，但各自占据不同的局部槽位；不同名称或参数则产生不同 layout。两种表示都会保留每个 placement。

In [7]:
same = build_cell()
syn = exp_synapse("shared")
same.place(at("soma", 0.31), syn)
same.place(at("soma", 0.39), syn)
same.init_state()

different = build_cell()
different.place(at("soma", 0.31), exp_synapse("left"))
different.place(at("soma", 0.39), exp_synapse("right"))
different.init_state()

comparison = [
    {"case": "same signature", "n_placements": len(same.point_placements), "n_layouts": len(same.layouts), "layout_sizes": [x.n_active for x in same.layouts]},
    {"case": "different names", "n_placements": len(different.point_placements), "n_layouts": len(different.layouts), "layout_sizes": [x.n_active for x in different.layouts]},
]
print(comparison)
display(pd.DataFrame(comparison))
assert comparison[0]["n_placements"] == comparison[1]["n_placements"] == 2
assert comparison[0]["n_layouts"] == 1
assert comparison[1]["n_layouts"] == 2

[{'case': 'same signature', 'n_placements': 2, 'n_layouts': 1, 'layout_sizes': [2]}, {'case': 'different names', 'n_placements': 2, 'n_layouts': 2, 'layout_sizes': [1, 1]}]


,case,n_placements,n_layouts,layout_sizes
0,same signature,2,1,[2]
1,different names,2,2,"[1, 1]"


## 5. branch 连接处可共享 point，同时保留各自 branch 归属

这里 `soma(1)` 与 `dend(0)` 是同一个电学连接点，但二者的原始 branch 和所属 CV 仍然不同。

In [8]:
cell = build_cell(with_dend=True)
clamp = braincell.CurrentClamp(
    delay=1.0 * u.ms,
    durations=2.0 * u.ms,
    amplitudes=0.1 * u.nA,
)
cell.place(at("soma", 1.0), clamp)
cell.place(at("dend", 0.0), clamp)

rows = placement_rows(cell)
print(rows)
display(pd.DataFrame(rows))
assert [row["branch_name"] for row in rows] == ["soma", "dend"]
assert [row["branch_x"] for row in rows] == [1.0, 0.0]
assert [row["cv_id"] for row in rows] == [0, 1]
assert len({row["point_id"] for row in rows}) == 1

[{'placement_id': 0, 'mechanism': 'CurrentClamp', 'branch_id': 0, 'branch_name': 'soma', 'branch_type': 'soma', 'branch_x': 1.0, 'cv_id': 0, 'cv_position': 'dist', 'point_id': 2}, {'placement_id': 1, 'mechanism': 'CurrentClamp', 'branch_id': 1, 'branch_name': 'dend', 'branch_type': 'basal_dendrite', 'branch_x': 0.0, 'cv_id': 1, 'cv_position': 'prox', 'point_id': 2}]


,placement_id,mechanism,branch_id,branch_name,branch_type,branch_x,cv_id,cv_position,point_id
0,0,CurrentClamp,0,soma,soma,1.0,0,dist,2
1,1,CurrentClamp,1,dend,basal_dendrite,0.0,1,prox,2


## 6. 共点 placement 拥有独立状态，电流在 point 上相加

两个 clamp 共享一个电压节点，但各自占据独立运行时槽位。状态可按 `placement_id` 查询；它们的电流会散射回共享 point 并相加。

In [9]:
cell = build_cell()
clamp = braincell.CurrentClamp(
    delay=1.0 * u.ms,
    durations=2.0 * u.ms,
    amplitudes=0.1 * u.nA,
)
cell.place(at("soma", 0.31), clamp)
cell.place(at("soma", 0.39), clamp)
cell.init_state()
layout = cell.layouts[0]

state_rows = [
    {
        "placement_id": placement.id,
        "point_id": placement.point_id,
        "delay_ms": float(np.asarray(cell.get_placement_state(placement.id)["delay"].to_decimal(u.ms))),
    }
    for placement in cell.point_placements
]
total_current = cell.runtime.evaluate_point_clamps(t=1.5 * u.ms).to_decimal(u.nA)
print(state_rows)
print("point current (nA):", np.asarray(total_current).tolist())
display(pd.DataFrame(state_rows))
assert layout.point_index.tolist() == [1, 1]
assert np.isclose(float(np.asarray(total_current)[1]), 0.2)

try:
    cell.get_point_state(1)
except ValueError as error:
    print("Ambiguous point query:", error)

[{'placement_id': 0, 'point_id': 1, 'delay_ms': 1.0}, {'placement_id': 1, 'point_id': 1, 'delay_ms': 1.0}]
point current (nA): [0.0, 0.20000000298023224, 0.0]


,placement_id,point_id,delay_ms
0,0,1,1.0
1,1,1,1.0


Ambiguous point query: Point 1 has multiple placements in layout 0; query by placement_id instead.


## 7. `by_post` 采样的是 placement，而不是去重后的电学 point

下面的突触后 pool 含有两个共点的突触 placement。设置 `replace=False` 后，两条入边会选中不同 placement 槽位，即使两个槽位映射到同一个 `point_id`。

In [10]:
post = build_cell(size=1)
syn = exp_synapse()
post.place(at("soma", 0.31), syn)
post.place(at("soma", 0.39), syn)
post.init_state()
layout = next(layout for layout, _ in post.runtime.iter_synapse_layouts())

edges = EdgeSet("E_to_I", "E", "I", [0, 1], [0, 0])
projection = Projection(
    name="E_to_I_exp",
    edges="E_to_I",
    synapse="exp",
    method=by_post(number=1, replace=False, seed=1),
)
connection = projection.to_connections(
    edges, pre_size=2, post_size=1, pool_size=layout.n_active
)[0]

contact_rows = []
for contact_id in range(connection.n_contact):
    synapse_index = int(connection.synapse_index[contact_id])
    placement_id = int(layout.placement_index[synapse_index])
    placement = post.get_point_placement(placement_id)
    contact_rows.append(
        {
            "contact_id": contact_id,
            "pre_index": int(connection.pre_index[contact_id]),
            "post_index": int(connection.post_index[contact_id]),
            "synapse_index": synapse_index,
            "placement_id": placement_id,
            "branch_x": placement.branch_x,
            "point_id": placement.point_id,
        }
    )

print(contact_rows)
display(pd.DataFrame(contact_rows))
assert {row["synapse_index"] for row in contact_rows} == {0, 1}
assert {row["placement_id"] for row in contact_rows} == {0, 1}
assert len({row["point_id"] for row in contact_rows}) == 1

[{'contact_id': 0, 'pre_index': 0, 'post_index': 0, 'synapse_index': 0, 'placement_id': 0, 'branch_x': 0.31, 'point_id': 1}, {'contact_id': 1, 'pre_index': 1, 'post_index': 0, 'synapse_index': 1, 'placement_id': 1, 'branch_x': 0.39, 'point_id': 1}]


,contact_id,pre_index,post_index,synapse_index,placement_id,branch_x,point_id
0,0,0,0,0,0,0.31,1
1,1,1,0,1,1,0.39,1


## 8. Locset 默认保留顺序和重复位置

`+` 是不去重的连接运算，因此两个相同位置会创建两个独立 placement；`|` 才是稳定去重的并集，`unique()` 也可显式去重。

In [11]:
cell = build_cell()
repeated_locations = at("soma", 0.5) + at("soma", 0.5)
cell.place(repeated_locations, exp_synapse())
rows = placement_rows(cell)
print(rows)
assert len(rows) == 2
assert [row["placement_id"] for row in rows] == [0, 1]
assert len({row["point_id"] for row in rows}) == 1
assert (at("soma", 0.5) | at("soma", 0.5)).evaluate(cell.morpho).points == ((0, 0.5),)

[{'placement_id': 0, 'mechanism': 'exp', 'branch_id': 0, 'branch_name': 'soma', 'branch_type': 'soma', 'branch_x': 0.5, 'cv_id': 0, 'cv_position': 'mid', 'point_id': 1}, {'placement_id': 1, 'mechanism': 'exp', 'branch_id': 0, 'branch_name': 'soma', 'branch_type': 'soma', 'branch_x': 0.5, 'cv_id': 0, 'cv_position': 'mid', 'point_id': 1}]


## 总结

| 声明方式 | 可处于同一 CV | 可处于同一 point | 可共享 layout | 实例相互独立 |
|---|---:|---:|---:|---:|
| 不同 `branch_x`、相同机制 | 是 | 是 | 是 | 是 |
| 重复完全相同的 `place()` | 是 | 是 | 是 | 是 |
| 一次 `place()` 中重复机制 | 是 | 是 | 是 | 是 |
| 不同机制名称或参数 | 是 | 是 | 否 | 是 |
| branch 连接处的两侧 | 否 | 是 | 是 | 是 |
| Locset `a + a` 的重复位置 | 是 | 是 | 是 | 是 |

电求解器使用 `point_id`；机制状态和网络靶向使用 placement 槽位；形态检查及后续空间采样则使用保留下来的 `branch_id + branch_x` 连续位置来源。